# Healthcare Analytics for Doctor Visits

This notebook performs a reproducible exploratory analysis of the supplied dataset. It focuses on descriptive patterns and does not infer causation.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/healthcare_doctor_visits.csv")
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


## 1. Data quality and structure

In [ ]:
display(df.info())
display(df.isna().sum().rename("missing_values").to_frame())
display(df.describe(include="all").T)


## 2. Doctor visit summary

In [ ]:
visits = df["visits"]
summary = pd.DataFrame({
    "metric": ["records", "mean visits", "median visits", "records with zero visits",
               "records with one or more visits", "maximum visits"],
    "value": [len(df), visits.mean(), visits.median(), (visits == 0).sum(),
              (visits >= 1).sum(), visits.max()]
})
display(summary)
print(f"Share with zero visits: {(visits == 0).mean():.1%}")
print(f"Share with one or more visits: {(visits >= 1).mean():.1%}")


In [ ]:
visit_counts = visits.value_counts().sort_index()
ax = visit_counts.plot(kind="bar", figsize=(8, 4), title="Distribution of doctor visits")
ax.set_xlabel("Number of visits")
ax.set_ylabel("Number of records")
plt.tight_layout()
plt.show()


## 3. Average visits by illness count

In [ ]:
illness_summary = df.groupby("illness", dropna=False)["visits"].agg(
    records="size", average_visits="mean", median_visits="median"
).reset_index()
display(illness_summary)
ax = illness_summary.plot(x="illness", y="average_visits", kind="bar", legend=False,
                           figsize=(8, 4), title="Average doctor visits by illness count")
ax.set_xlabel("Reported illness count")
ax.set_ylabel("Average visits")
plt.tight_layout()
plt.show()


## 4. Compare groups

In [ ]:
group_columns = ["gender", "private", "freepoor", "freerepat", "nchronic", "lchronic"]
for col in group_columns:
    print(f"\nAverage visits by {col}")
    display(df.groupby(col, dropna=False)["visits"].agg(
        records="size", average_visits="mean", median_visits="median"
    ).reset_index())


## 5. Interpretation and limitations

- Most records report zero visits in the period represented by the dataset.
- Average visits increase across some illness-count groups in this sample.
- Group differences are descriptive associations only; they do not prove causation.
- Check the source documentation for the meaning and coding of `age`, `health`, and the observation period before making stronger claims.
- Do not use these results to make individual medical decisions.
